# Semantical annotations with Azure OpenAI client

In [1]:
import sys
sys.path.append("../")

In [2]:
import os
import json
import openai
import logging
import sqlite3

from pandas import read_csv
from tqdm.auto import tqdm 
from configparser import ConfigParser

from llm_library.openai import configure_azure_client
from llm_library.prompt_templating import UserPrompt
from llm_library.prompt_templating import SystemPrompt
from llm_library.prompt_templating import AssistantPrompt
from llm_library.prompt_templating import InputList

from estnltk import Text
from estnltk.taggers import VabamorfAnalyzer

## I. Set up Azure API 

In [3]:
config = ConfigParser()
status = config.read('../../model_configurations/azure_gpt-4.ini') 
assert status == ['../../model_configurations/azure_gpt-4.ini']

In [4]:
client = configure_azure_client(config)

## II. Annotate words in the context with GPT  

In [5]:
PROMPT_PATH = 'prompts'
INPUT_FILE = 'data/lausetega_100_sone.csv'

CHUNK_SIZE = 5
MODEL = config['azure-configuration']['model']

OUTPUT_FILE = f'results/run_01_{MODEL}.db'

In [6]:
tasks = read_csv(INPUT_FILE)[['lemma', 'sentence_id', 'head_id', 'loc', 'text']]
assert len(tasks) == 500, "Fix current state"

We need to add the wordform as it is not part of the input.

In [7]:
tasks['wordform'] = ''
morph_analyzer = VabamorfAnalyzer()

#
for i, row in tasks.iterrows():
    text = Text(row['text']).tag_layer('words')
    wordform = text['words'][row['loc'] - 1].text
    
    # print(wordform)
    lemmas = list(map(lambda x: x['lemma'], morph_analyzer.analyze_token(wordform)))

    # Check that we get the correct wordform
    if row['lemma'] not in lemmas:
        if row['lemma'] == 'Ju' and 'Just' not in lemmas:
            pass
        else: 
            print(text)
            print(wordform)
            print(row['lemma'])
            print(row)
            assert False, 'Something went wrong with indexing'

    tasks.loc[i, 'wordform'] = wordform

In [8]:
connection = sqlite3.connect(OUTPUT_FILE)
cursor = connection.cursor()
cursor.execute("DROP TABLE IF EXISTS  input_meta")
cursor.execute("DROP TABLE IF EXISTS  model_annotations")

# Create a table for modified input
cursor.execute(
"""
CREATE TABLE input_meta
(
    lemma TEXT,
    wordform TEXT,
    sentence_id INTEGER,
    head_id INTEGER,
    loc INTEGER
);
""")


# Create a table for modified input
cursor.execute(
"""
CREATE TABLE model_annotations
(
    input TEXT,
    model TEXT,
    semantic_class TEXT,
    response TEXT
);
""")

connection.commit()

# Fill in the input meta data before starting queries with ChatGPT
tasks[['lemma', 'wordform', 'sentence_id', 'head_id', 'loc']].to_sql('input_meta', connection, if_exists="append", index=False)
connection.commit()

In [11]:
# Warning supression?
# import http.client
# http.client.HTTPConnection.debuglevel = 0

In [9]:
logging.getLogger("openai").setLevel(logging.WARNING)

for filename in os.listdir(PROMPT_PATH):
    if not filename.endswith(".txt"):
        continue
    semantic_class =filename.split('.')[0]
    print(f'Semantic class: {semantic_class}')

    file_path = os.path.join(PROMPT_PATH, filename)
    with open(file_path, "r", encoding="utf-8") as f:
        system_prompt = f.read()
    print('Prompt text')
    print(system_prompt)
    print('------------------------------------------------------------------------------------------')
    
    for i in tqdm(range(0, len(tasks), CHUNK_SIZE)):
        chunk = tasks[i:i + CHUNK_SIZE]

        # Prepare user prompt
        user_prompt = json.dumps([
            {
                'word': word, 
                'context': context
            } for _, (word, context) in chunk[['wordform', 'text']].iterrows()
        ], indent=2)

        # Query GPT
        try:
            response = client.chat.completions.create(
                model = config['azure-configuration']['deployment_name'],
                messages = [
                    SystemPrompt(system_prompt),
                    UserPrompt(user_prompt)
                ]
            )
        except Exception as e:
            # Store the cause of an abnormal execution
            print(e)
            cursor.execute(
                """
                INSERT INTO model_annotations (input, model, semantic_class, response)
                VALUES (?, ?, ?, ?)
                """,
                (user_prompt, MODEL, semantic_class, 'FAIL')
            )              
            continue
            

        # Store results of normal execution
        cursor.execute(
            """
            INSERT INTO model_annotations (input, model, semantic_class, response)
            VALUES (?, ?, ?, ?)
            """,
            (user_prompt, MODEL, semantic_class, response.choices[0].message.content)
        )        
        connection.commit()
        
connection.close()

Semantic class: belongings
Prompt text
Provide the output in JSON format, where:

'1' indicates that the given Estonian word is a noun referring to a physical item no bigger than a human that a person can own.
'0' indicates that the word does not meet these criteria.

Evaluate the following words according to whether they serve that purpose in the given context.

------------------------------------------------------------------------------------------


  0%|          | 0/100 [00:00<?, ?it/s]

INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-ope

  0%|          | 0/100 [00:00<?, ?it/s]

INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-ope

...

INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-ope

  0%|          | 0/100 [00:00<?, ?it/s]

INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-ope

  0%|          | 0/100 [00:00<?, ?it/s]

INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-openai-api-management.azure-api.net/ltat-tartunlp/openai/deployments/gpt4-sven/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
INFO:_client.py:1025: HTTP Request: POST https://tu-ope

In [10]:
print("We are done!")

We are done!
